<!-- mirandastech-aula-v2 -->

# Laboratório — Ridge, Lasso e Elastic Net

Este laboratório compara regressões penalizadas em dados sintéticos com features correlacionadas e escalas muito diferentes. O conjunto de teste é separado **antes** da seleção; `alpha` é escolhido somente por validação cruzada no desenvolvimento.

[Voltar para a aula](../aulas/05-regularizacao-ridge-lasso-elastic-net.md)

## Objetivos e protocolo

1. criar um problema cuja estrutura geradora seja conhecida;
2. comparar baseline, OLS, Ridge, Lasso e Elastic Net;
3. selecionar hiperparâmetros sem consultar o teste;
4. inspecionar caminhos, esparsidade e estabilidade;
5. verificar a invariância à troca de unidade quando há padronização.

**Unidade de análise:** uma observação sintética independente. **Seed:** `20260908`. Em dados temporais ou agrupados, o particionamento deveria respeitar essa estrutura.

## Ambiente

Dependências mínimas verificáveis:

```text
Python >= 3.10
numpy >= 1.24
pandas >= 2.0
matplotlib >= 3.7
scikit-learn >= 1.4
```

No Colab, essas bibliotecas já costumam estar disponíveis. Não instalamos pacotes automaticamente para não alterar silenciosamente o ambiente.

In [ ]:
import platform
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SEED = 20260908
np.set_printoptions(precision=6, suppress=True)

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)

## 1. Dados sintéticos com correlação e escalas distintas

Cinco fatores latentes geram o sinal. As primeiras features observadas são aproximações ruidosas desses fatores; pares redundantes criam multicolinearidade. Outras colunas contêm apenas ruído. Algumas unidades são multiplicadas por 100 ou divididas por 100, tornando a padronização indispensável.

In [ ]:
rng = np.random.default_rng(SEED)
n = 420
z = rng.normal(size=(n, 5))

signal_features = np.column_stack([
    z[:, 0] + rng.normal(0, 0.08, n),
    z[:, 0] + rng.normal(0, 0.08, n),  # quase duplicada
    z[:, 1] + rng.normal(0, 0.12, n),
    z[:, 1] + rng.normal(0, 0.12, n),  # quase duplicada
    100 * (z[:, 2] + rng.normal(0, 0.10, n)),
    0.01 * (z[:, 3] + rng.normal(0, 0.10, n)),
    z[:, 4] + rng.normal(0, 0.15, n),
])

noise_scales = np.logspace(-1, 1, 13)
noise_features = rng.normal(size=(n, len(noise_scales))) * noise_scales
X = np.column_stack([signal_features, noise_features])
feature_names = [f"sinal_{i}" for i in range(signal_features.shape[1])] + [
    f"ruido_{i}" for i in range(noise_features.shape[1])
]

y = 5.0 * z[:, 0] - 3.2 * z[:, 1] + 2.0 * z[:, 2] + 1.3 * z[:, 3] + rng.normal(0, 2.8, n)

assert X.shape == (420, 20)
assert np.isfinite(X).all() and np.isfinite(y).all()
print("Shape de X:", X.shape)
print("Correlação sinal_0 × sinal_1:", f"{np.corrcoef(X[:, 0], X[:, 1])[0, 1]:.6f}")
print("Razão entre maior e menor desvio-padrão:", f"{X.std(axis=0).max() / X.std(axis=0).min():.1f}×")

## 2. Teste reservado e validação cruzada

O teste é isolado primeiro. Todos os valores de `alpha` são avaliados nos 75% de desenvolvimento. Como os dados são independentes por construção, usamos `KFold` embaralhado com seed fixa.

In [ ]:
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED
)
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
alphas = np.logspace(-3, 2, 16)

assert len(X_dev) == 315 and len(X_test) == 105
assert alphas[0] == 1e-3 and alphas[-1] == 1e2
print("Desenvolvimento:", X_dev.shape, "| teste reservado:", X_test.shape)
print("Grade de alpha:", alphas)

## 3. Seleção de `alpha` apenas no desenvolvimento

Cada candidato é um pipeline completo: o scaler é reajustado dentro de cada fold. Para tornar a comparação transparente, fazemos a grade manualmente. Elastic Net usa `l1_ratio=0.5`; selecionar também essa razão exigiria uma grade bidimensional dentro do mesmo protocolo.

In [ ]:
def pipeline_for(kind, alpha=None):
    if kind == "OLS":
        estimator = LinearRegression()
    elif kind == "Ridge":
        estimator = Ridge(alpha=alpha)
    elif kind == "Lasso":
        estimator = Lasso(alpha=alpha, max_iter=100_000, tol=1e-7, random_state=SEED)
    elif kind == "Elastic Net":
        estimator = ElasticNet(
            alpha=alpha, l1_ratio=0.5, max_iter=100_000, tol=1e-7, random_state=SEED
        )
    else:
        raise ValueError(kind)
    return make_pipeline(StandardScaler(), estimator)


def cv_curve(kind):
    rows = []
    for alpha in alphas:
        scores = -cross_val_score(
            pipeline_for(kind, alpha), X_dev, y_dev,
            scoring="neg_root_mean_squared_error", cv=cv, n_jobs=1,
        )
        rows.append({
            "modelo": kind,
            "alpha": alpha,
            "rmse_cv": scores.mean(),
            "erro_padrao": scores.std(ddof=1) / np.sqrt(len(scores)),
        })
    return pd.DataFrame(rows)


curves = pd.concat([cv_curve(name) for name in ["Ridge", "Lasso", "Elastic Net"]], ignore_index=True)
best = curves.loc[curves.groupby("modelo")["rmse_cv"].idxmin()].sort_values("modelo")
print(best.to_string(index=False, float_format=lambda value: f"{value:.6f}"))

assert set(best["modelo"]) == {"Ridge", "Lasso", "Elastic Net"}
assert best["alpha"].between(alphas.min(), alphas.max()).all()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for name, group in curves.groupby("modelo"):
    ax.plot(group["alpha"], group["rmse_cv"], marker="o", label=name)
ax.set_xscale("log")
ax.set_xlabel("alpha")
ax.set_ylabel("RMSE médio na validação cruzada")
ax.set_title("Curvas de validação — somente dados de desenvolvimento")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

## 4. Reajuste e única avaliação no teste

Agora cada método penalizado usa seu `alpha` selecionado. O conjunto reservado participa somente desta célula. A média do desenvolvimento serve como baseline. Reportamos RMSE, MAE e $R^2$; a escolha já foi feita por RMSE de validação.

In [ ]:
selected_alpha = dict(zip(best["modelo"], best["alpha"]))
models = {"OLS": pipeline_for("OLS")}
for name in ["Ridge", "Lasso", "Elastic Net"]:
    models[name] = pipeline_for(name, selected_alpha[name])

results = [{
    "modelo": "Baseline média",
    "rmse_teste": mean_squared_error(y_test, np.full_like(y_test, y_dev.mean())) ** 0.5,
    "mae_teste": mean_absolute_error(y_test, np.full_like(y_test, y_dev.mean())),
    "r2_teste": r2_score(y_test, np.full_like(y_test, y_dev.mean())),
}]

for name, model in models.items():
    model.fit(X_dev, y_dev)
    prediction = model.predict(X_test)
    results.append({
        "modelo": name,
        "rmse_teste": mean_squared_error(y_test, prediction) ** 0.5,
        "mae_teste": mean_absolute_error(y_test, prediction),
        "r2_teste": r2_score(y_test, prediction),
    })

results = pd.DataFrame(results).sort_values("rmse_teste")
print(results.to_string(index=False, float_format=lambda value: f"{value:.6f}"))

baseline_rmse = results.loc[results["modelo"] == "Baseline média", "rmse_teste"].iloc[0]
assert results.iloc[0]["rmse_teste"] < 0.55 * baseline_rmse
assert np.isfinite(results.select_dtypes("number")).all().all()

## 5. Coeficientes na escala padronizada e esparsidade

Como todos os modelos receberam features padronizadas, podemos comparar magnitudes dos coeficientes. Ainda assim, magnitude não é causalidade. Para Lasso e Elastic Net, declaramos coeficiente ativo quando $|eta_j|>10^{-8}$.

In [ ]:
coeficients = pd.DataFrame({
    name: model[-1].coef_ for name, model in models.items()
}, index=feature_names)

active = (coeficients.abs() > 1e-8).sum().rename("coeficientes_ativos")
print(active.to_string())
print("\nDez maiores magnitudes em Lasso:")
print(coeficients.assign(abs_lasso=coeficients["Lasso"].abs()).nlargest(10, "abs_lasso").drop(columns="abs_lasso").round(4))

assert active["Ridge"] == X.shape[1]
assert active["Lasso"] < X.shape[1]
assert active["Lasso"] >= 4

## 6. Caminhos de regularização

As curvas abaixo mostram o que uma métrica agregada não revela. Em Ridge, os pesos encolhem continuamente. Em Lasso, vários atingem zero. Features redundantes podem disputar o mesmo sinal.

In [ ]:
path_alphas = np.logspace(-3, 2, 40)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True)

for ax, name in zip(axes, ["Ridge", "Lasso"]):
    paths = []
    for alpha in path_alphas:
        candidate = pipeline_for(name, alpha).fit(X_dev, y_dev)
        paths.append(candidate[-1].coef_)
    paths = np.asarray(paths)
    for j in range(paths.shape[1]):
        ax.plot(path_alphas, paths[:, j], alpha=0.75)
    ax.axvline(selected_alpha[name], color="black", linestyle="--", label="alpha selecionado")
    ax.set_xscale("log")
    ax.set_title(name)
    ax.set_xlabel("alpha")
    ax.grid(alpha=0.2)
    ax.legend()
axes[0].set_ylabel("Coeficiente na escala padronizada")
fig.suptitle("Caminhos de regularização")
plt.tight_layout()
plt.show()

## 7. Teste de invariância à unidade

Representamos `sinal_0` em uma unidade 1.000 vezes maior. Sem scaler, Ridge aplica uma penalidade efetivamente diferente. Com `StandardScaler` no pipeline, as previsões devem permanecer iguais, salvo erro numérico.

In [ ]:
X_dev_unit = X_dev.copy()
X_test_unit = X_test.copy()
X_dev_unit[:, 0] *= 1_000
X_test_unit[:, 0] *= 1_000

alpha_demo = 10.0
ridge_raw_a = Ridge(alpha=alpha_demo).fit(X_dev, y_dev)
ridge_raw_b = Ridge(alpha=alpha_demo).fit(X_dev_unit, y_dev)
raw_difference = np.max(np.abs(ridge_raw_a.predict(X_test) - ridge_raw_b.predict(X_test_unit)))

ridge_scaled_a = make_pipeline(StandardScaler(), Ridge(alpha=alpha_demo)).fit(X_dev, y_dev)
ridge_scaled_b = make_pipeline(StandardScaler(), Ridge(alpha=alpha_demo)).fit(X_dev_unit, y_dev)
scaled_difference = np.max(np.abs(ridge_scaled_a.predict(X_test) - ridge_scaled_b.predict(X_test_unit)))

print("Diferença máxima sem padronização:", f"{raw_difference:.9f}")
print("Diferença máxima com padronização:", f"{scaled_difference:.3e}")

assert raw_difference > 1e-3
assert scaled_difference < 1e-10

## 8. Estabilidade por reamostragem

Reamostramos as linhas de desenvolvimento com reposição. Em cada amostra, reajustamos OLS e um Ridge moderado. Com colunas quase duplicadas, OLS tende a redistribuir o peso de maneira mais volátil. A comparação usa coeficientes na escala padronizada.

In [ ]:
rng_boot = np.random.default_rng(SEED + 1)
b = 160
bootstrap_coefs = {"OLS": [], "Ridge(alpha=10)": []}
stability_models = {
    "OLS": make_pipeline(StandardScaler(), LinearRegression()),
    "Ridge(alpha=10)": make_pipeline(StandardScaler(), Ridge(alpha=10.0)),
}

for _ in range(b):
    idx = rng_boot.integers(0, len(X_dev), len(X_dev))
    for name, prototype in stability_models.items():
        fitted = clone(prototype).fit(X_dev[idx], y_dev[idx])
        bootstrap_coefs[name].append(fitted[-1].coef_)

stability = pd.DataFrame({
    name: np.asarray(values).std(axis=0, ddof=1).mean()
    for name, values in bootstrap_coefs.items()
}, index=["desvio-padrão médio dos coeficientes"]).T
ratio = stability.loc["OLS"].iloc[0] / stability.loc["Ridge(alpha=10)"].iloc[0]
print(stability.to_string(float_format=lambda value: f"{value:.6f}"))
print("OLS / Ridge:", f"{ratio:.3f}×")

assert ratio > 1.2

## 9. Verificações finais

Os `asserts` anteriores verificaram shapes, separação, esparsidade, desempenho, invariância de unidade e estabilidade. Esta última célula resume as escolhas que tornam o experimento reproduzível.

In [ ]:
checks = {
    "seed_fixa": SEED,
    "teste_separado_antes_da_busca": True,
    "scaler_dentro_do_pipeline": True,
    "folds": cv.get_n_splits(),
    "teste_avaliado_apos_selecao": True,
    "outputs_do_commit": "limpos após validar uma cópia",
}
print(pd.Series(checks).to_string())
assert all(checks[key] for key in [
    "teste_separado_antes_da_busca", "scaler_dentro_do_pipeline", "teste_avaliado_apos_selecao"
])

## Conclusões

- Ridge estabilizou coeficientes sem produzir esparsidade.
- Lasso eliminou variáveis, mas a escolha entre substitutas correlacionadas exige cautela.
- Elastic Net combinou contração $L_2$ e seleção $L_1$.
- A padronização dentro do pipeline tornou o resultado invariável à troca de unidade.
- `alpha` foi selecionado somente no desenvolvimento; o teste permaneceu reservado.

Experimente trocar `l1_ratio`, aumentar o número de features de ruído ou reduzir a amostra. Faça qualquer nova escolha no desenvolvimento e preserve um novo teste para a estimativa final.